# 12 — Symptom severity: Cam-CAN vs HCPHow much clinical signal is there to predict, in each cohort? Three figures and two tables.**Read this before the numbers.** The two cohorts are measured with different instruments, whosepublished cutoffs were drawn for different purposes:| | instrument | range | bands | what the first cutoff means ||---|---|---|---|---|| Cam-CAN | HADS, per subscale | 0–21 | Normal / Mild / Moderate / Severe at 0–7 / 8–10 / 11–14 / 15–21 | a **screening** line, chosen for sensitivity || HCP | ASR, DSM-oriented | raw, reported as T | Normal / Borderline / Clinical at T<65 / 65–69 / ≥70 | the **93rd percentile** of a reference sample, by construction |So `% mild-or-worse` in Cam-CAN and `% borderline-or-worse` in HCP are **not comparable**. A gapbetween them is mostly a gap between two conventions. The comparison that *is* valid lives in F3and in the percentile table: within-cohort percentiles, the floor fraction, and the spread.**Bands are described on T; the model is fitted on raw.** A T-score has age and sex normed out ofit in the reference sample, which HADS never had. Regressing a T-score against brain states wouldquietly remove age and sex from one cohort and not the other — exactly the dimension this designis about.

In [ ]:
import sysfrom pathlib import PathREPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path[:0] = [str(REPO / "tools"), str(REPO / "notebooks")]import pandas as pdimport symptom_distribution as sd      # bands, id matching, load()  -- stdlib onlyimport symptom_plots as sp             # the three figures# ---- point these at your own copies -----------------------------------------HCP_PHENO    = Path("/project/6008063/tamires/cohorts/hcp_pheno/participants_pheno.csv")HCP_SUBJECTS = REPO / "config" / "hcp7t_fix_participants.csv"CAMCAN_DIR   = Path("/project/6008063/tamires/cohorts/camcan/dataman/useraccess/opendata/"                    "paule_toussaint_camcan01870")CAMCAN_SUBJECTS = REPO / "config" / "camcan_participants.csv"SAVE_DIR     = None        # e.g. REPO / "notebooks" / "figures" / "12_symptoms"# -----------------------------------------------------------------------------d = sd.load(HCP_PHENO, HCP_SUBJECTS, CAMCAN_DIR, CAMCAN_SUBJECTS)for coh, blk in d.items():    print(f"{coh:7s} n={blk['n']:4d}  {blk['instrument']:18s} scales={list(blk['scales'])}")    if blk["missing_pheno"]:        print(f"         !! {len(blk['missing_pheno'])} imaging subjects with no phenotype row")

## Table 1 — bands, absolute and percent`Zero` is split out of the lowest band. That split needs no per-subject join: a raw score of 0 isbelow every cutoff on every scale here, so a zero-scoring subject is necessarily in the lowest band.

In [ ]:
bands = pd.DataFrame(sp.band_table(d))wide = (bands.assign(cell=lambda x: x["n"].astype(str) + "  (" + x["pct"].round(1).astype(str) + "%)")             .pivot(index=["cohort", "construct"], columns="band", values="cell")             .fillna("—"))display(wide)bands

## Table 2 — the comparable numbersPercentiles within each cohort, the fraction sitting at the floor, and the spread. `sd_over_max` isblank for the ASR because that scale has no published maximum here.**`pct_at_floor` is the number that decides whether a target is usable.** Spearman over a columnwhere a large share of subjects tie at zero has far less power than its *n* suggests, and moresubjects do not fix a target that is mostly floor.

In [ ]:
summ = pd.DataFrame(sp.summary_table(d))display(summ)

## F1 — the raw distributions, each on its own axisSeparate axes on purpose. Dashed rules are the HADS cutoffs; the ASR panels carry none, because theASR's cutoffs are defined on T and have no position on a raw axis.

In [ ]:
_ = sp.fig_histograms(d, save_dir=SAVE_DIR)

## F2 — band compositionThe figure a reader will screenshot without the notebook around it, so the caveat is printed on it.

In [ ]:
_ = sp.fig_bands(d, save_dir=SAVE_DIR)

## F3 — Q-Q: the honest side-by-sideOne point per percentile: its *x* is the score Cam-CAN reaches there, its *y* what HCP reaches. Noshared axis and no assumed maximum, so this is the comparison the band percentages cannot make.How to read it: a **flat run** along either axis is that cohort's floor. A curve that **risessteeply** means HCP's scale spreads faster than Cam-CAN's at those percentiles; **shallow** means theopposite. The line is neutral grey, not a cohort colour, because it is the relationship between thetwo — not either one of them.

In [ ]:
_ = sp.fig_quantiles(d, save_dir=SAVE_DIR)

## The other HCP scales`DSM_Anxi` and `DSM_Depr` pair with the two HADS subscales, so they lead. But `ASR_Intn`(internalizing: anxious/depressed + withdrawn + somatic) has **more items**, hence a lower floor —which can make it the better regression target even though it has no HADS counterpart.Compare the `pct_at_floor` column below against Table 2 before choosing.

In [ ]:
extra = pd.DataFrame(sp.summary_table(d, constructs=("anxious/depressed", "internalizing")))display(extra)

## Families, for `--group-col`HCP contains twins and siblings. With *n* = 168 this stops being a footnote: two members of a familyin different CV folds leak. The counts below say how many groups a `GroupKFold` by `Family_ID` wouldactually have, and whether any cluster is larger than a pair.

In [ ]:
from collections import Counterrows, _, _ = sd._hcp_rows(HCP_PHENO, HCP_SUBJECTS)fam = Counter((r.get("Family_ID") or "").strip() for r in rows if (r.get("Family_ID") or "").strip())sizes = Counter(fam.values())print(f"subjects : {sum(fam.values())}")print(f"families : {len(fam)}")print(f"cluster sizes: {dict(sorted(sizes.items()))}")print(f"-> GroupKFold by Family_ID has {len(fam)} groups; largest cluster {max(sizes)} subject(s)")print(f"-> one-per-family instead would leave {len(fam)} subjects, which is the n=37 regime "      f"this cohort was expanded to escape" if len(fam) < 60 else "")